# Volume 2, Chapter 6: Rings and Ideals (Python companion)

Exercises first, solutions at the end. We implement polynomial arithmetic over $\Z/p\Z$ from scratch, watch unique factorisation fail in $\Z[\sqrt{-5}]$, factor $X^4+1$ modulo every small prime, find which imaginary quadratic rings are Euclidean for the norm, and use the Chinese remainder theorem the way computer algebra systems do: **multi-modular computation**, which computes an exact integer determinant by working modulo many small primes.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
import time
import numpy as np
import sympy

## Exercises

**P6.1 (★) Polynomials over $\Z/p\Z$.** Represent a polynomial by its list of coefficients (constant first). Write addition, multiplication, Euclidean division and gcd over $\Z/p\Z$. Check that $X^p-X=\prod_{a\in\Z/p\Z}(X-a)$ for $p=7$, and compute $\gcd(X^{12}-1,X^8-1)$ over $\Z/5\Z$.

In [ ]:
# Your code here


**P6.2 (★) Factorisations in $\Z[\sqrt{-5}]$.** Find all elements of norm at most $50$ that are irreducible (no divisor of smaller norm $>1$). Then list, for $n\le 50$, the integers that have two genuinely different factorisations into irreducibles in $\Z[\sqrt{-5}]$ (Proposition 6.11).

In [ ]:
# Your code here


**P6.3 (★★) $X^4+1$ modulo $p$.** Using P6.1, factor $X^4+1$ over $\Z/p\Z$ for the primes $p<60$ and record the degrees of the factors. Relate the pattern to $p\bmod8$ (Exercise 6.8).

In [ ]:
# Your code here


**P6.4 (★★) Which rings are norm-Euclidean?** The ring of integers $\mathcal O$ of $\Q(\sqrt{-d})$ is a lattice in $\C$ (Volume 1, Chapter 11). By Exercise 6.4, it is Euclidean for $N=|\cdot|^2$ as soon as every point of the plane is at distance $<1$ from the lattice. Compute this *covering radius* numerically for the squarefree $d\le 30$ and list the $d$ where it is $<1$.

In [ ]:
# Your code here


**P6.5 (★★★) CS application: multi-modular determinants.** The determinant of a $40\times40$ matrix with entries in $[-100,100]$ is an integer of about $100$ digits. Gaussian elimination over $\Q$ produces huge fractions. Instead:
1. bound $|\det A|$ by Hadamard's inequality $|\det A|\le\prod_i\|\text{row}_i\|$;
2. compute $\det A\bmod p$ for enough $30$-bit primes $p$ that their product exceeds twice the bound;
3. recover $\det A$ by the Chinese remainder theorem (Theorem 6.9 for $R=\Z$), choosing the representative in $(-M/2,M/2]$.

Compare with SymPy's exact determinant, and time both.

In [ ]:
# Your code here


---
## Solutions

### P6.1

In [ ]:
def trim(f):
    while f and f[-1] == 0:
        f.pop()
    return f

def padd(f, g, p):
    n = max(len(f), len(g))
    return trim([((f[i] if i < len(f) else 0) + (g[i] if i < len(g) else 0)) % p for i in range(n)])

def pmul(f, g, p):
    if not f or not g:
        return []
    h = [0] * (len(f) + len(g) - 1)
    for i, a in enumerate(f):
        for j, b in enumerate(g):
            h[i + j] = (h[i + j] + a * b) % p
    return trim(h)

def pdivmod(f, g, p):
    f = f[:]; q = [0] * max(len(f) - len(g) + 1, 0)
    inv = pow(g[-1], -1, p)
    while len(f) >= len(g) and f:
        c = f[-1] * inv % p; k = len(f) - len(g)
        q[k] = c
        for i, b in enumerate(g):
            f[i + k] = (f[i + k] - c * b) % p
        trim(f)
    return trim(q), f

def pgcd(f, g, p):
    while g:
        f, g = g, pdivmod(f, g, p)[1]
    inv = pow(f[-1], -1, p)
    return [a * inv % p for a in f]          # monic

p = 7
prod = [1]
for a in range(p):
    prod = pmul(prod, [(-a) % p, 1], p)
print('prod (X - a) =', prod, ' X^7 - X =', trim([0, p - 1] + [0] * 5 + [1]))
xn = lambda n, p: trim([p - 1] + [0] * (n - 1) + [1])     # X^n - 1
print('gcd(X^12 - 1, X^8 - 1) over F_5:', pgcd(xn(12, 5), xn(8, 5), 5))

$\prod_a(X-a)$ has coefficients $[0,6,0,0,0,0,0,1]$, that is $X^7-X$ (as $6=-1$): this is Fermat's little theorem turned into a polynomial identity. The gcd of $X^{12}-1$ and $X^8-1$ is $X^4-1$, as in $\Z[X]$: $\gcd(X^m-1,X^n-1)=X^{\gcd(m,n)}-1$ over any field, by the same Euclidean algorithm as on the exponents.

### P6.2

In [ ]:
def norm(a, b):
    return a * a + 5 * b * b

elems = [(a, b) for a in range(-8, 9) for b in range(-4, 5) if 1 < norm(a, b) <= 50]

def divides(x, y):
    # does x divide y in Z[sqrt(-5)]?  y / x = y * conj(x) / N(x)
    (a, b), (c, d) = x, y
    n = norm(a, b)
    re, im = c * a + 5 * d * b, d * a - c * b
    return re % n == 0 and im % n == 0

def irreducible(x):
    n = norm(*x)
    return not any(divides(y, x) for y in elems if 1 < norm(*y) < n)

irr = sorted({x for x in elems if irreducible(x)}, key=lambda x: norm(*x))
print('norms of irreducibles up to 50:', sorted({norm(*x) for x in irr}))
print('norms 2 and 3 occur:', any(norm(*x) in (2, 3) for x in elems))

def factorisations(n, start=0):
    # multisets of irreducibles (up to sign) whose norms multiply to n^2 and whose product is n
    out = []
    reps = sorted({max(x, (-x[0], -x[1])) for x in irr}, key=lambda x: (norm(*x), x))
    def mul(x, y):
        return (x[0] * y[0] - 5 * x[1] * y[1], x[0] * y[1] + x[1] * y[0])
    def rec(target_norm, i, prod, chosen):
        if target_norm == 1:
            if prod in ((n, 0), (-n, 0)):
                out.append(tuple(chosen))
            return
        for j in range(i, len(reps)):
            m = norm(*reps[j])
            if target_norm % m == 0:
                rec(target_norm // m, j, mul(prod, reps[j]), chosen + [reps[j]])
    rec(n * n, 0, (1, 0), [])
    return out

def show(x):
    a, b = x
    return f'{a}' if b == 0 else f'({a}{b:+d}√-5)'

for n in range(2, 51):
    f = factorisations(n)
    if len(f) > 1:
        print(n, ' = '.join('·'.join(show(x) for x in fac) for fac in f))

No element has norm $2$, $3$, $7$, $11$ or $13$: these integers are not of the form $a^2+5b^2$, so the irreducibles $2$, $3$, $7$, ... cannot split. Several integers have two factorisations: $6=2\cdot3=(1+\sqrt{-5})(1-\sqrt{-5})$, $9=3\cdot3=(2+\sqrt{-5})(2-\sqrt{-5})$, $14=2\cdot7=(3+\sqrt{-5})(3-\sqrt{-5})$, $21$ in three ways, and products such as $18$ or $42$ in many more (the list shows each factorisation up to signs). In Volume 1, Chapter 11, each of these is explained by the factorisation of the *ideals* $(2)$, $(3)$, $(7)$ into prime ideals, which is unique.

### P6.3

In [ ]:
def factor_mod_p(f, p):
    # brute-force factorisation into monic irreducibles of small degree (enough for degree 4)
    factors = []
    for d in (1, 2):
        for coeffs in itertools.product(range(p), repeat=d):
            g = list(coeffs) + [1]
            while len(f) > 1:
                q, r = pdivmod(f, g, p)
                if r:
                    break
                factors.append(d); f = q
    if len(f) > 1:
        factors.append(len(f) - 1)
    return sorted(factors)

for p in sympy.primerange(2, 60):
    print(p, p % 8, factor_mod_p([1, 0, 0, 0, 1], p))

For $p=2$ we get four linear factors ($(X+1)^4$). For $p\equiv1\pmod8$, four linear factors: then $\Z/p\Z$ contains a primitive $8$th root of unity (its multiplicative group is cyclic of order divisible by $8$), and the roots of $X^4+1$ are exactly the primitive $8$th roots of unity. For $p\equiv3,5,7\pmod8$, two quadratic factors, never irreducible, as Exercise 6.8 proves. The splitting pattern depends only on $p\bmod8$: this is the first instance of the link between splitting of primes and Galois groups (Chapter 10; here the group is $(\Z/8\Z)^*$).

### P6.4

In [ ]:
def covering_radius(d, grid=400):
    # lattice O_K for Q(sqrt(-d)): basis 1, w with w = sqrt(-d) or (1 + sqrt(-d)) / 2
    w = complex(0.5, math.sqrt(d) / 2) if d % 4 == 3 else complex(0, math.sqrt(d))
    best = 0.0
    for s in np.linspace(0, 1, grid):
        for t in np.linspace(0, 1, grid // 4):
            z = s + t * w           # a point of the fundamental parallelogram
            dist = min(abs(z - (m + n * w)) for m in (-1, 0, 1, 2) for n in (-1, 0, 1, 2))
            best = max(best, dist)
    return best

squarefree = [d for d in range(1, 31) if all(d % (q * q) for q in range(2, 6))]
euclid = []
for d in squarefree:
    r = covering_radius(d)
    if r < 1:
        euclid.append(d)
    print(f'd={d:2d}  covering radius ~ {r:.3f}' + ('   < 1' if r < 1 else ''))
print('norm-Euclidean by the covering argument:', euclid)

The covering radius is below $1$ exactly for $d=1,2,3,7,11$: these five imaginary quadratic rings of integers are Euclidean for the norm, and it is known that they are the only ones (for $d=19,43,67,163$ the ring is a PID without being Euclidean, Dummit and Foote §8.2). For $d=5$ the radius is $\sqrt{3/2}\approx1.22$, matching Exercise 6.4(c). The grid search can only underestimate the radius, so the values above $1$ are certain. For the others, the exact radius is the circumradius of a triangle of lattice points, for instance $3/\sqrt{11}\approx0.905$ for $d=11$ (triangle $0,1,\frac{1+\sqrt{-11}}2$, with sides $1,\sqrt3,\sqrt3$), which the grid reproduces.

### P6.5

In [ ]:
def det_mod_p(A, p):
    A = [[x % p for x in row] for row in A]
    n = len(A); det = 1
    for c in range(n):
        piv = next((r for r in range(c, n) if A[r][c]), None)
        if piv is None:
            return 0
        if piv != c:
            A[c], A[piv] = A[piv], A[c]; det = -det
        det = det * A[c][c] % p
        inv = pow(A[c][c], -1, p)
        for r in range(c + 1, n):
            f = A[r][c] * inv % p
            if f:
                A[r] = [(x - f * y) % p for x, y in zip(A[r], A[c])]
    return det % p

def det_multimodular(A):
    bound = math.prod(math.isqrt(sum(x * x for x in row)) + 1 for row in A)   # Hadamard
    M, residues, moduli = 1, [], []
    p = 2**30
    while M <= 2 * bound:
        p = sympy.prevprime(p)
        residues.append(det_mod_p(A, p)); moduli.append(p); M *= p
    x = sympy.ntheory.modular.crt(moduli, residues)[0]
    return (x - M if x > M // 2 else x), len(moduli)

random.seed(6)
n = 40
A = [[random.randint(-100, 100) for _ in range(n)] for _ in range(n)]
t = time.perf_counter(); d1, k = det_multimodular(A); t1 = time.perf_counter() - t
t = time.perf_counter(); d2 = sympy.Matrix(A).det(method='bareiss'); t2 = time.perf_counter() - t
print('digits:', len(str(abs(d1))), ' primes used:', k, ' agree:', d1 == d2)
print(f'multi-modular {t1:.2f} s   SymPy Bareiss {t2:.2f} s')

The two determinants agree. The multi-modular method does only arithmetic on small numbers: each $\det A\bmod p$ costs $n^3/3$ operations on $30$-bit integers, and a dozen primes suffice. Fraction-free elimination (Bareiss), the exact method SymPy uses, works with integers that grow to the size of the answer at every step. The idea, ``compute modulo many primes, then glue with the Chinese remainder theorem'', is used throughout computer algebra (gcds of polynomials, linear system solving, resultants), and it is Theorem 6.9 at work: $\Z/M\Z\cong\prod\Z/p_i\Z$ is a ring isomorphism, so ring computations can be done componentwise.